In [1]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("CineMatch AI Model - Ready!")


CineMatch AI Model - Ready!


In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

print("Movies dataset:", movies.shape)
print("Credits dataset:", credits.shape)

movies.head()


Movies dataset: (4803, 20)
Credits dataset: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [3]:
credits = credits.rename(columns={"movie_id": "id"})

movies = movies.merge(
    credits[["id", "cast", "crew"]],
    on="id",
    how="left"
)

def parse_list(value):
    try:
        items = ast.literal_eval(value)
        return [
            item["name"].lower().replace(" ", "")
            for item in items
        ]
    except (ValueError, TypeError, SyntaxError):
        return []

movies["overview"] = movies["overview"].fillna("")
movies["genres_list"] = movies["genres"].apply(parse_list)
movies["keywords_list"] = movies["keywords"].apply(parse_list)

movies = movies.drop_duplicates(subset="title")
movies = movies.reset_index(drop=True)

print("Movies processed:", len(movies))
movies[["title", "genres_list", "keywords_list"]].head()

Movies processed: 4800


,title,genres_list,keywords_list
0,Avatar,"[action, adventure, fantasy, sciencefiction]","[cultureclash, future, spacewar, spacecolony, ..."
1,Pirates of the Caribbean: At World's End,"[adventure, fantasy, action]","[ocean, drugabuse, exoticisland, eastindiatrad..."
2,Spectre,"[action, adventure, crime]","[spy, basedonnovel, secretagent, sequel, mi6, ..."
3,The Dark Knight Rises,"[action, crime, drama, thriller]","[dccomics, crimefighter, terrorist, secretiden..."
4,John Carter,"[action, adventure, sciencefiction]","[basedonnovel, mars, medallion, spacetravel, p..."


In [4]:
movies["tags"] = (
    movies["genres_list"].apply(lambda x: " ".join(x * 3))
    + " "
    + movies["keywords_list"].apply(lambda x: " ".join(x * 2))
    + " "
    + movies["overview"].str.lower()
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

matrix = vectorizer.fit_transform(movies["tags"])

print("AI recommendation model ready!")
print("Number of movies:", len(movies))
print("Feature matrix shape:", matrix.shape)

AI recommendation model ready!
Number of movies: 4800
Feature matrix shape: (4800, 5000)


In [5]:
def recommend_movies(movie_title, top_n=5):
    matches = movies[
        movies["title"].str.lower() == movie_title.lower()
    ]

    if matches.empty:
        print("Movie dataset mein nahi mili!")
        return

    movie_index = matches.index[0]

    scores = cosine_similarity(
        matrix[movie_index], matrix
    ).flatten()

    similar_indices = scores.argsort()[::-1]
    similar_indices = [
        i for i in similar_indices if i != movie_index
    ][:top_n]

    results = movies.iloc[similar_indices][
        ["title", "vote_average", "release_date"]
    ].copy()

    results["similarity"] = [
        round(float(scores[i]), 3) for i in similar_indices
    ]

    return results.reset_index(drop=True)


recommend_movies("Krrish")

,title,vote_average,release_date,similarity
0,The Helix... Loaded,4.8,2005-01-01,0.381
1,Timecop,5.5,1994-09-15,0.249
2,Echo Dr.,5.0,2013-11-08,0.232
3,The Time Machine,5.8,2002-03-04,0.220
4,Containment,5.1,2015-07-09,0.219


In [6]:
recommend_movies("Yeh Jawaani Hai Deewani")

,title,vote_average,release_date,similarity
0,Moonlight Mile,6.5,2002-09-09,0.201
1,Breaking Upwards,5.6,2009-03-14,0.186
2,Coffee Town,6.7,2013-07-09,0.176
3,About Last Night,6.0,2014-02-14,0.165
4,The Lovers,4.8,2015-02-13,0.150


In [7]:
recommend_movies("Fiza")

,title,vote_average,release_date,similarity
0,Housefull,5.2,2010-04-30,0.497
1,Tango,6.8,1998-02-12,0.495
2,The Circle,6.6,2000-09-08,0.457
3,Live-In Maid,7.8,2004-09-21,0.448
4,Water,6.8,2005-09-08,0.404
